# Lecture Bot

Ask questions about the documents in `docs/` (placeholders for now: syllabus and module
description). The switch decides how the documents reach the model:

- **Context stuffing:** every question is sent together with all pages. Misses nothing, but
  everything must fit into the model's context window.
- **RAG** (Retrieval-Augmented Generation): a search picks the pages closest in meaning to the
  question, and only those are sent. Scales to large collections, but can miss the right page.

In [ ]:
import json
import subprocess
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import ollama

from prompt_view import show_prompt

mode = widgets.ToggleButtons(options=[("Context stuffing", "stuffing"), ("RAG", "rag")],
                             value="stuffing")
mode

## 1. Load the documents

In [ ]:
pages = []   # (source, text), one entry per PDF page
for pdf in sorted(Path("docs").glob("*.pdf")):
    text = subprocess.run(["pdftotext", "-layout", str(pdf), "-"],
                          capture_output=True, text=True, check=True).stdout
    pages += [(f"{pdf.name} p.{i}", t) for i, t in enumerate(text.split("\f"), 1) if t.strip()]

for source, text in pages:
    print(f"{source}: ~{len(text) // 4} tokens")

<details>
<summary><b>What happens here</b></summary>

- Models read text, not PDFs. `pdftotext` extracts the text; images and layout are lost.
- `pdftotext` marks each page break with `\f`, so splitting there gives one entry per page.
  For slides, a page is one slide: a natural chunk, with no chunk size to choose.
- A *token* is a word piece, about 4 characters. Characters divided by 4 is an estimate.
</details>

## 2. Prepare the search (RAG only)

In [ ]:
def embed(texts):
    return np.array(ollama.embed(model="qwen3-embedding:0.6b", input=texts).embeddings)

vectors = embed([text for _, text in pages])

def retrieve(question, k=3):
    q = embed(["Instruct: Given a question, retrieve lecture pages that answer it\nQuery: "
               + question])[0]
    scores = vectors @ q / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
    return [(*pages[i], scores[i]) for i in scores.argsort()[::-1][:k]]

<details>
<summary><b>How the search works</b></summary>

1. `embed()` turns every page into a vector (a list of numbers) with the embedding model
   `qwen3-embedding:0.6b`. Similar meaning gives similar vectors, even across languages: an
   English question finds a German page.
2. `retrieve()` embeds the question, with an instruction line in front because the model was
   trained that way, and scores every page by *cosine similarity*: 1 means same meaning,
   0 means unrelated. The best `k=3` pages are sent; 3 is a common starting value, not tuned.

Why not more? Two popular additions did not pay off here:

- **Keyword search** (BM25) helps when questions and documents share a language. With English
  questions on German pages, it pushed the right page down in our test.
- **Letting the model rewrite the question first** did not help 7B models in recent studies,
  and it costs a second model call per question.

With only 4 placeholder pages, RAG sends almost everything. The difference shows once the real
lecture is in `docs/`.
</details>

## 3. Ask

In [ ]:
INSTRUCTIONS = """You are a teaching assistant for the module "Agentic AI".
Answer only from the lecture material below. If it does not cover the question,
say so. Name the source you used. Answer in English."""

def ask(question):
    global last
    if mode.value == "stuffing":
        sources = [(source, text, None) for source, text in pages]
    else:
        sources = retrieve(question)
    context = "\n\n".join(f"=== {source} ===\n{text}" for source, text, _ in sources)
    messages = [{"role": "system", "content": INSTRUCTIONS + "\n\n" + context},
                {"role": "user", "content": question}]

    response = ollama.chat(model="qwen2.5:7b", messages=messages, options={"temperature": 0})

    last = {"mode": mode.value, "model": "qwen2.5:7b", "sources": sources, "messages": messages,
            "answer": response.message.content, "tokens": response.prompt_eval_count}
    print(response.message.content)

<details>
<summary><b>Prompt, explained</b></summary>

- A call is a list of messages. The **system** message holds the rules plus the context, the
  **user** message holds the question.
- The instructions each have a job:
  - *Answer only from the material*: prevents *hallucination*, confidently invented answers.
  - *If not covered, say so*: gives the model a permitted way out.
  - *Name the source*: makes the answer checkable.
- Every question stands alone. The model has no memory, and we send no chat history, so
  follow-ups like "and after that?" do not work. That keeps the input short and the search
  precise.
- `temperature: 0` always picks the most likely token, so answers are reproducible.
- There is no context window setting: Ollama uses the model's maximum, 32,768 tokens for
  `qwen2.5:7b`.
</details>

In [ ]:
ask("When is MCP covered, and what is it about?")

In [ ]:
ask("Which weeks cover reasoning models?")

In [ ]:
ask("What is the name of the professor's dog?")

<details>
<summary><b>What the three questions test</b></summary>

1. A lookup: the answer is in the syllabus.
2. A second lookup, phrased differently from the syllabus.
3. Not in the material: the bot should say so, not invent a name.
</details>

In [ ]:
# Your questions:

## 4. What the model received

Replays your last question: every block of text the model read, in order. Click a block to
read it in full. Run the cell again to replay.

In [ ]:
show_prompt(last)

## 5. Under the hood: what goes into Ollama

`ollama.chat()` hides two steps. First, the Python client sends your messages to the Ollama
server as an HTTP request. Second, Ollama turns the messages into one plain text, which is
what the model actually reads.

In [ ]:
# Step 1: the HTTP request the Python client sends
print("POST http://localhost:11434/api/chat")
print(json.dumps({"model": last["model"], "messages": last["messages"],
                  "options": {"temperature": 0}, "stream": False},
                 indent=2, ensure_ascii=False))

In [ ]:
# Step 2: the text the model reads, built with qwen2.5's chat template
def to_raw(messages):
    turns = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages)
    return turns + "<|im_start|>assistant\n"

raw = to_raw(last["messages"])
print(raw)

In [ ]:
# Proof: send the raw text ourselves, with Ollama's template switched off
check = ollama.generate(model=last["model"], prompt=raw, raw=True, options={"temperature": 0})
print("same token count:", check.prompt_eval_count == last["tokens"])
print("same answer:     ", check.response.strip() == last["answer"].strip())

<details>
<summary><b>How the raw text works</b></summary>

- A language model only continues text. The chat is an illusion made from plain text.
- `<|im_start|>` and `<|im_end|>` are *special tokens*: markers the model learned in training
  to mean "a message starts" and "a message ends". Each is a single token.
- The text ends with `<|im_start|>assistant`, so the most likely continuation is the
  assistant's answer. The model stops when it writes `<|im_end|>`.
- The template is **model-specific**. Llama or Gemma use different markers for the same idea.
  Run `ollama show qwen2.5:7b --template` in a terminal to see the original.
- `raw=True` tells Ollama to skip its template. If token count and answer match, `to_raw()`
  rebuilt the model input exactly.
</details>

<details>
<summary><b>Which mode, and is this state of the art?</b></summary>

- **Documents fit into the window:** use stuffing. For small collections it is the
  recommended default; Anthropic, for example, advises skipping RAG below about 200,000
  tokens.
- **Documents do not fit:** use RAG. Production systems often add keyword search and a
  *reranker*, a second model that re-sorts the top pages. Ollama cannot run a reranker yet.
- **Letting the model search in a loop** (agentic RAG) did not beat a plain search with 7B
  models in recent studies, unless the model was specially trained for it. Lab 09 covers RAG
  in depth.
- *Prompt injection* is not another word for stuffing: it names an attack where text inside
  a document overrides the instructions (Lab 13).
</details>